# IAN 620 — Module 8: Large Language Models and Applications

This notebook is a practical companion to Module 8.

We use one small local model throughout the notebook: **Qwen/Qwen3-0.6B**.

The notebook follows this path:

**environment → tokens → token IDs → logits → probabilities → generation → chat → prompting → structured output → evaluation → LoRA → RAG/tool bridge**

## 1. Create and select a Python environment

Before installing the LLM libraries, use a **separate Python environment** for this notebook. An isolated environment keeps this project's packages separate from your system Python and other projects.

**Recommended Python version: 3.11**

Choose **one** method: Python `venv` or Conda. You do not need both.

<details>
<summary><strong>Windows — Python venv</strong></summary>

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
```

For Command Prompt, activate with:

```bat
.venv\Scripts\activate.bat
```
</details>

<details>
<summary><strong>macOS or Linux — Python venv</strong></summary>

```bash
python3 -m venv .venv
source .venv/bin/activate
```
</details>

<details>
<summary><strong>Anaconda or Miniconda</strong></summary>

```bash
conda create -n ian620-module8 python=3.11 -y
conda activate ian620-module8
```
</details>

<details>
<summary><strong>VS Code and Jupyter kernel</strong></summary>

1. Open the project folder in VS Code.
2. Create and activate the environment in **Terminal → New Terminal**.
3. Run **Python: Select Interpreter** and choose `.venv` or `ian620-module8`.
4. Open this notebook and choose the same environment from the kernel selector in the upper-right corner.
5. If it does not appear, install `ipykernel` inside the environment.

```bash
python -m pip install ipykernel
python -m ipykernel install --user --name ian620-module8 --display-name "Python (IAN 620 Module 8)"
```
</details>

> After creating the environment, select it as the notebook kernel before installing the course libraries.

### Optional — create `.venv` from a notebook cell

Run the next cell only when working locally. It creates `.venv`, installs `ipykernel`, and registers a Jupyter kernel. The notebook cannot automatically switch itself into the new environment, so select **Python (IAN 620 Module 8)** after the cell finishes.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    print("Colab already provides an isolated environment.")
else:
    env_dir = Path(".venv")

    if not env_dir.exists():
        subprocess.check_call([sys.executable, "-m", "venv", str(env_dir)])
        print("Created .venv")
    else:
        print(".venv already exists")

    if os.name == "nt":
        env_python = env_dir / "Scripts" / "python.exe"
    else:
        env_python = env_dir / "bin" / "python"

    subprocess.check_call([str(env_python), "-m", "pip", "install", "--upgrade", "pip"])
    subprocess.check_call([str(env_python), "-m", "pip", "install", "ipykernel"])
    subprocess.check_call([
        str(env_python), "-m", "ipykernel", "install", "--user",
        "--name", "ian620-module8",
        "--display-name", "Python (IAN 620 Module 8)"
    ])

    print("Select this kernel: Python (IAN 620 Module 8)")

### Verify the active environment

In [ ]:
import sys
import platform

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Operating system:", platform.system())
print("Machine:", platform.machine())

## 2. Install the libraries

Install these packages **after selecting the environment as the notebook kernel**.

In [ ]:
%pip install -q transformers==5.20.1 accelerate==1.15.0 peft==0.21.2 pandas matplotlib

## 3. Imports and device check

In [ ]:
import json
import time
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

## 4. Load a small local LLM

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()
print("Loaded:", MODEL_NAME)

# Part 1 — Tokens and Token IDs

An LLM does not read text directly. The tokenizer converts text into tokens and integer token IDs.

In [ ]:
text = "The customer submitted a refund request."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Text:", text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("Decoded:", tokenizer.decode(token_ids))

# Part 2 — Logits and Next-Token Probabilities

In [ ]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

next_logits = outputs.logits[0, -1]
probabilities = torch.softmax(next_logits, dim=-1)
top_probabilities, top_ids = torch.topk(probabilities, k=10)

rows = []
for probability, token_id in zip(top_probabilities, top_ids):
    rows.append({
        "token": tokenizer.decode([token_id.item()]),
        "token_id": token_id.item(),
        "probability": float(probability)
    })

probability_df = pd.DataFrame(rows)
probability_df

In [ ]:
probability_df.plot.bar(
    x="token",
    y="probability",
    legend=False,
    title="Top Next-Token Probabilities"
)

# Part 3 — Generation

In [ ]:
prompt = "Machine learning is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

### Compare temperature values

In [ ]:
prompt = "Write one sentence describing artificial intelligence."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

for temperature in [0.3, 0.7, 1.2]:
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    print(f"\nTEMPERATURE = {temperature}")
    print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# Part 4 — Chat Models

In [ ]:
def ask_llm(
    user_message,
    system_message="You are a concise and helpful assistant.",
    max_new_tokens=180,
    temperature=0.0
):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    settings = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id
    }

    if temperature > 0:
        settings.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": 0.95
        })
    else:
        settings["do_sample"] = False

    with torch.no_grad():
        output_ids = model.generate(**inputs, **settings)

    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0].strip()

In [ ]:
print(ask_llm("Explain tokenization with one simple example."))

# Part 5 — Prompt Engineering and Structured Output

In [ ]:
json_prompt = '''
Read the support message and return JSON.

Message:
I was charged twice and need a refund.

Required fields:
- category
- urgency
- summary

Return JSON only.
'''

json_response = ask_llm(json_prompt, max_new_tokens=120)
print(json_response)

In [ ]:
try:
    parsed = json.loads(json_response)
    print("Valid JSON:", parsed)
except json.JSONDecodeError:
    parsed = None
    print("The response is not valid JSON.")

# Part 6 — Evaluation

In [ ]:
evaluation_data = pd.DataFrame([
    {"text": "I was charged twice.", "expected": "Billing"},
    {"text": "I cannot sign in.", "expected": "Login"},
    {"text": "Where is the user manual?", "expected": "Other"}
])

predictions = []

for text in evaluation_data["text"]:
    prompt = f'''
Classify the support message as exactly one of:
Billing
Login
Other

Message: {text}
Return only the label.
'''
    predictions.append(ask_llm(prompt, max_new_tokens=20).strip())

evaluation_data["prediction"] = predictions
evaluation_data["correct"] = (
    evaluation_data["prediction"].str.lower()
    == evaluation_data["expected"].str.lower()
)

print(f"Accuracy: {evaluation_data['correct'].mean():.1%}")
evaluation_data

# Part 7 — LoRA Concept

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

print(lora_config)

# Part 8 — Bridge to RAG

In [ ]:
question = "What is the refund deadline?"

retrieved_documents = [
    {"source_id": "S1", "text": "Refund requests must be submitted within 14 days of delivery."},
    {"source_id": "S2", "text": "Customers can submit refund requests from the Orders page."}
]

context = "\n\n".join(
    f"[{doc['source_id']}] {doc['text']}"
    for doc in retrieved_documents
)

rag_prompt = f'''
Use only the context below.

Context:
{context}

Question:
{question}

Give a concise answer and cite the source IDs.
'''

print(ask_llm(rag_prompt, max_new_tokens=100))

# Part 9 — Bridge to Tool Calling

In [ ]:
tool_prompt = '''
Choose one tool:
- search_policy: company policies
- lookup_order: a specific order

User request: What is the return policy?

Return JSON only:
{"tool": "...", "reason": "..."}
'''

tool_response = ask_llm(tool_prompt, max_new_tokens=100)
print(tool_response)

# Module 8 Summary

This notebook covered environment setup, tokenization, logits, generation, chat templates, prompting, structured output, evaluation, LoRA, RAG grounding, and a bridge to tool calling.